In [22]:
import json
import joblib
import numpy as np
from sentence_transformers import SentenceTransformer


# load model
model = joblib.load("svm_model.pkl")

# load feature columns
with open("feature_columns.json", "r") as f:
    feature_columns = json.load(f)

# load embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

scaler = joblib.load("scaler.pkl")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [66]:
def take_input():

    movie_title = input("Enter movie title: ")

    print("\nSelect genres (type names separated by comma)")
    print("Available: (no genres listed), Animation, Crime, Film-Noir, Musical, Mystery, Romance, Thriller, War, Western")

    user_input = input("Enter genres: ")
    selected_genres = [g.strip() for g in user_input.split(",")]

    all_genres = [
        '(no genres listed)', 'Animation', 'Crime', 'Film-Noir', 'Musical',
        'Mystery', 'Romance', 'Thriller', 'War', 'Western'
    ]

    genres = {g: 1 if g in selected_genres else 0 for g in all_genres}

    user_rating_count = float(input("\nUser rating count: "))
    movie_avg_rating = float(input("Movie average rating: "))
    

    return movie_title, genres, user_rating_count, movie_avg_rating

In [67]:
def build_input(movie_title, genres, user_rating_count, movie_avg_rating):

    # 1. Genre features (already aligned with feature_columns order assumption)
    genre_vector = np.array(list(genres.values()))

    # 2. Scale numeric features
    numeric = np.array([[user_rating_count, movie_avg_rating]])
    scaled_numeric = scaler.transform(numeric)[0]

    # 3. Movie embedding (FIXED)
    emb = embedder.encode([movie_title])[0]

    # 4. Final feature vector
    X = np.hstack([
        genre_vector,
        emb,
        scaled_numeric
    ])

    return X.reshape(1, -1)

In [68]:
def run_system():

    movie_title, genres, user_rating_count, movie_avg_rating= take_input()

    X = build_input(movie_title, genres, user_rating_count, movie_avg_rating)

    pred = model.predict(X)[0]

    if pred == 1:
        print("\n Recommended")
    else:
        print("\n Not Recommended")

In [69]:
import warnings
warnings.filterwarnings("ignore")

In [70]:
run_system()

Enter movie title:  Toy Story (1995) 



Select genres (type names separated by comma)
Available: (no genres listed), Animation, Crime, Film-Noir, Musical, Mystery, Romance, Thriller, War, Western


Enter genres:   Animation, Children, Comedy

User rating count:  3000
Movie average rating:  4.7



 Recommended


In [71]:
run_system() 

Enter movie title:   Horror Night (2020)



Select genres (type names separated by comma)
Available: (no genres listed), Animation, Crime, Film-Noir, Musical, Mystery, Romance, Thriller, War, Western


Enter genres:   Western

User rating count:  5
Movie average rating:  1.0



 Not Recommended
